In [1]:
from faker import Faker
import random
from datetime import datetime, timedelta
import pandas as pd

In [2]:
fake = Faker()
Faker.seed(0)
random.seed(0)

NUM_USERS = 500
NUM_CONTACT_REQUESTS = 500

In [3]:
roles = ['customer', 'admin']
statuses = ['NEW', 'IN_PROGRESS', 'RESOLVED']

with open('address_data.txt', 'r', encoding='utf-8') as f:
    address_lines = [line.strip() for line in f.readlines()]

with open('subject_message_data.txt', 'r', encoding='utf-8') as f:
    subject_msg_lines = [line.strip() for line in f.readlines()]
    subject_messages = [line.split(":", 1) for line in subject_msg_lines if ":" in line]


In [ ]:
users_data = []
for i in range(1, NUM_USERS + 1):
    addr_line = address_lines[i % len(address_lines)]
    addr_parts = [p.strip() for p in addr_line.split(',')]

    address = addr_parts[0] if len(addr_parts) >= 1 else 'Default Address'
    city = addr_parts[1] if len(addr_parts) >= 2 else 'Default City'
    state = addr_parts[2] if len(addr_parts) >= 3 else 'Default State'
    country = addr_parts[3] if len(addr_parts) >= 4 else 'Unknown'
    postcode = addr_parts[4] if len(addr_parts) >= 5 else '00000'

    dob = fake.date_of_birth(minimum_age=15, maximum_age=65)
    created_at = fake.date_time_this_decade()
    updated_at = created_at + timedelta(days=random.randint(0, 30))

    users_data.append({
        'id': i,
        'first_name': fake.first_name(),
        'last_name': fake.last_name(),
        'address': address[:70],
        'city': city[:40],
        'state': state[:40],
        'country': country[:40],
        'postcode': postcode[:10],
        'phone': fake.msisdn()[:11], 
        'dob': dob.strftime('%Y-%m-%d'),
        'email': fake.unique.email()[:60],
        'password': fake.password(length=12),
        'role': random.choice(roles),
        'enabled': True,
        'failed_login_attempts': random.randint(0, 5),
        'created_at': created_at.isoformat(),
        'updated_at': updated_at.isoformat()
    })


In [5]:
contact_requests_data = []
for i in range(1, NUM_CONTACT_REQUESTS + 1):
    subj, msg = subject_messages[i % len(subject_messages)]
    created_at = fake.date_time_this_year()
    updated_at = created_at + timedelta(minutes=random.randint(1, 120))

    contact_requests_data.append({
        'id': i,
        'user_id': random.choice(users_data)['id'],
        'name': fake.name(),
        'email': fake.email(),
        'subject': subj.strip()[:120],
        'message': msg.strip()[:250],
        'status': random.choice(statuses),
        'created_at': created_at.isoformat(),
        'updated_at': updated_at.isoformat()
    })


In [6]:
df_users = pd.DataFrame(users_data)
df_contacts = pd.DataFrame(contact_requests_data)

with pd.ExcelWriter("22127444_DataGeneration_Data.xlsx") as writer:
    df_users.to_excel(writer, sheet_name="Users", index=False)
    df_contacts.to_excel(writer, sheet_name="ContactRequests", index=False)